# Momentum Decay with Break Point Modelling

In [1]:
import numpy as np

# ─────────────────────────────────────────
#  BETA MODEL
# ─────────────────────────────────────────

class BetaModel:
    """
    Tracks a single probability using a Beta distribution
    with exponential decay on in-match evidence.

    alpha_prior / beta_prior  : fixed career anchor, never touched after init
    alpha_match / beta_match  : in-match component, decayed each observation

    During warmup (n_obs < warmup threshold), returns pure prior.
    After warmup, blends prior + in-match.
    """

    def __init__(self, career_rate, prior_strength, lam=0.95, warmup=None):
        self.alpha_prior = career_rate * prior_strength
        self.beta_prior  = (1.0 - career_rate) * prior_strength

        self.alpha_match = 0.0
        self.beta_match  = 0.0

        self.lam    = lam
        self.warmup = warmup if warmup is not None else int(1 / (1 - lam))
        self.n_obs  = 0

    def get_p(self):
        if self.n_obs < self.warmup:
            return self.alpha_prior / (self.alpha_prior + self.beta_prior)
        total = (self.alpha_prior + self.beta_prior +
                 self.alpha_match + self.beta_match)
        return (self.alpha_prior + self.alpha_match) / total

    def update(self, won):
        self.alpha_match *= self.lam
        self.beta_match  *= self.lam
        if won:
            self.alpha_match += 1.0
        else:
            self.beta_match  += 1.0
        self.n_obs += 1

    def reset(self):
        self.alpha_match = 0.0
        self.beta_match  = 0.0
        self.n_obs       = 0


# ─────────────────────────────────────────
#  PLAYER
# ─────────────────────────────────────────

class Player:
    """
    Holds all Beta models for one player.

    Stats expected:
      first_in              : 1st serve in %
      win_first             : win % on 1st serve points (serving)
      win_second            : win % on 2nd serve points (serving)
      return_first          : win % returning opponent's 1st serve
      return_second         : win % returning opponent's 2nd serve
      bp_save_rate          : break point saved %
      bp_save_faced         : career break points faced (used as prior strength)
      bp_convert_rate       : break point converted %
      bp_convert_opps       : career break point opportunities (prior strength)
    """

    def __init__(self, name, stats, prior_strength=200, lam=0.95):
        self.name  = name
        self.stats = stats

        # Serve models — prior strength is tuned hyperparameter
        self.serve_first_model  = BetaModel(stats['win_first'],
                                            prior_strength, lam)
        self.serve_second_model = BetaModel(stats['win_second'],
                                            prior_strength, lam)

        # Return models
        self.return_first_model  = BetaModel(stats['return_first'],
                                             prior_strength, lam)
        self.return_second_model = BetaModel(stats['return_second'],
                                             prior_strength, lam)

        # Break point models — prior strength = actual observed count
        # warmup = 20 break point situations (rare, so kept at 20)
        self.bp_save_model = BetaModel(
            career_rate    = stats['bp_save_rate'],
            prior_strength = stats['bp_save_faced'],
            lam            = lam,
            warmup         = 20
        )
        self.bp_convert_model = BetaModel(
            career_rate    = stats['bp_convert_rate'],
            prior_strength = stats['bp_convert_opps'],
            lam            = lam,
            warmup         = 20
        )

    def reset(self):
        self.serve_first_model.reset()
        self.serve_second_model.reset()
        self.return_first_model.reset()
        self.return_second_model.reset()
        self.bp_save_model.reset()
        self.bp_convert_model.reset()


# ─────────────────────────────────────────
#  BREAK POINT DETECTION
# ─────────────────────────────────────────

def is_break_point(server_pts, receiver_pts):
    """
    Returns True if the current score is a break point situation
    (receiver is one point from winning the game).

    Score encoding: 0=0, 1=15, 2=30, 3=40, 4+=deuce/adv logic
    We track raw point counts and handle deuce separately.
    """
    # Receiver needs 4+ points and leads by 1 or more from deuce
    # Break point: receiver at 40 (3pts) and server < 40, OR receiver has adv
    if receiver_pts == 3 and server_pts < 3:
        return True
    # Advantage receiver (both reached deuce, receiver leads)
    if receiver_pts >= 4 and server_pts >= 3 and receiver_pts == server_pts + 1:
        return True
    return False


# ─────────────────────────────────────────
#  POINT SIMULATION
# ─────────────────────────────────────────

def sim_point(server: Player, receiver: Player,
              server_pts: int, receiver_pts: int):
    """
    Simulate one point.
    Uses break point models when score is a break point situation.
    Updates all relevant Beta models after outcome.
    Returns True if server wins.
    """
    bp = is_break_point(server_pts, receiver_pts)
    first_in = server.stats['first_in']

    if bp:
        # Break point: blend save % vs convert %
        p_save    = server.bp_save_model.get_p()
        p_convert = receiver.bp_convert_model.get_p()
        p_win = (p_save + (1.0 - p_convert)) / 2.0

        # Simulate outcome (no serve-type split on break points)
        server_won = np.random.random() < p_win

        # Update break point models for both players
        server.bp_save_model.update(server_won)
        receiver.bp_convert_model.update(not server_won)

        # Also update serve/return models — break points are still
        # serve points and count toward those career tendencies
        if np.random.random() < first_in:
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)

    else:
        # Normal point: blend serve % vs return %
        p_serve_1st  = server.serve_first_model.get_p()
        p_serve_2nd  = server.serve_second_model.get_p()
        p_return_1st = receiver.return_first_model.get_p()
        p_return_2nd = receiver.return_second_model.get_p()

        p_win_1st = (p_serve_1st + (1.0 - p_return_1st)) / 2.0
        p_win_2nd = (p_serve_2nd + (1.0 - p_return_2nd)) / 2.0

        if np.random.random() < first_in:
            server_won = np.random.random() < p_win_1st
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server_won = np.random.random() < p_win_2nd
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)

    return server_won


# ─────────────────────────────────────────
#  GAME SIMULATION
# ─────────────────────────────────────────

def sim_game(server: Player, receiver: Player):
    """
    Simulate one game. Returns True if server wins.
    Tracks raw point counts to detect break point situations.
    """
    # Raw point counts (not tennis notation)
    score = [0, 0]   # [server_pts, receiver_pts]

    while True:
        server_won = sim_point(server, receiver,
                               server_pts   = score[0],
                               receiver_pts = score[1])

        if server_won:
            score[0] += 1
        else:
            score[1] += 1

        # Win: 4+ points, lead by 2
        if score[0] >= 4 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 4 and score[1] - score[0] >= 2:
            return False


# ─────────────────────────────────────────
#  TIEBREAK SIMULATION
# ─────────────────────────────────────────

def sim_tiebreak(p1: Player, p2: Player, p1_serves_first: bool):
    """
    Simulate a tiebreak. First to 7, win by 2.
    No break point logic in tiebreaks (no break points exist).
    Server alternates: 1 point then every 2.
    """
    score = [0, 0]
    point_count = 0

    while True:
        if point_count == 0:
            p1_serves = p1_serves_first
        else:
            p1_serves = p1_serves_first == (point_count % 2 == 0)

        server   = p1 if p1_serves else p2
        receiver = p2 if p1_serves else p1

        # Tiebreak points are never break points — pass impossible score
        server_won = sim_point(server, receiver,
                               server_pts=0, receiver_pts=0)
        p1_won = server_won if p1_serves else not server_won

        if p1_won:
            score[0] += 1
        else:
            score[1] += 1

        point_count += 1

        if score[0] >= 7 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 7 and score[1] - score[0] >= 2:
            return False


# ─────────────────────────────────────────
#  SET SIMULATION
# ─────────────────────────────────────────

def sim_set(p1: Player, p2: Player, p1_serves_first: bool):
    """
    Simulate one set. Returns (True if p1 wins, p1_serving next set).
    """
    games = [0, 0]
    p1_serving = p1_serves_first

    while True:
        server   = p1 if p1_serving else p2
        receiver = p2 if p1_serving else p1

        server_won  = sim_game(server, receiver)
        p1_won_game = server_won if p1_serving else not server_won

        if p1_won_game:
            games[0] += 1
        else:
            games[1] += 1

        p1_serving = not p1_serving

        # Tiebreak at 6-6
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1, p2, p1_serving)
            if p1_won_tb:
                games[0] += 1
            else:
                games[1] += 1
            return (games[0] > games[1]), p1_serving

        if games[0] >= 6 and games[0] - games[1] >= 2:
            return True, p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2:
            return False, p1_serving


# ─────────────────────────────────────────
#  MATCH SIMULATION
# ─────────────────────────────────────────

def sim_match(p1: Player, p2: Player,
              p1_serves_first: bool = True,
              best_of: int = 3):
    """
    Simulate one full match. Resets both players at start.
    Returns True if p1 wins.
    """
    p1.reset()
    p2.reset()

    sets_needed = best_of // 2 + 1
    sets = [0, 0]
    p1_serving = p1_serves_first

    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1, p2, p1_serving)
        if p1_won_set:
            sets[0] += 1
        else:
            sets[1] += 1

    return sets[0] > sets[1]


# ─────────────────────────────────────────
#  RUN SIMULATION
# ─────────────────────────────────────────

def run_simulation(p1: Player, p2: Player,
                   N: int = 100_000,
                   p1_serves_first: bool = True,
                   best_of: int = 3):
    wins = 0
    for _ in range(N):
        if sim_match(p1, p2, p1_serves_first, best_of):
            wins += 1

    p1_prob = wins / N
    p2_prob = 1 - p1_prob

    p1_se = np.sqrt(p1_prob * p2_prob / N)
    p2_se = np.sqrt(p2_prob * p1_prob / N)  # same value, symmetric

    return p1_prob, p1_se, p2_prob, p2_se



In [25]:
# ── INPUTS ─────────────────────────────────────────
P1_NAME    = "Daniel Merida"
P2_NAME    = "Tomas Barrios Vera"
COURT_TYPE = "Clay"   # "Hard", "Clay", "Grass", "Carpet", or "all"
YEAR       = 2026     # stats will be fetched for YEAR - 1

In [ ]:
import glob
import os
import pandas as pd

def load_player_stats(full_name, court_type, year, csv_dir):
    pattern = os.path.join(csv_dir, "player_stats_*.csv")
    files = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(f"No player_stats_*.csv found in {csv_dir}")
    csv_path = max(files, key=os.path.getmtime)

    df = pd.read_csv(csv_path)

    player_df = df[df["full_name"].str.strip().str.lower() == full_name.strip().lower()]
    if player_df.empty:
        available = sorted(df["full_name"].dropna().unique().tolist())
        raise ValueError(f"Player '{full_name}' not found. Sample available: {available[:10]}")

    target_year = year
    stat_cols = [
        "FirstServePercentage", "FirstServePointsWonPercentage",
        "SecondServePointsWonPercentage", "FirstServeReturnPointsWonPercentage",
        "SecondServeReturnPointsWonPercentage", "BreakPointsSavedPercentage",
        "BreakPointsFaced", "BreakPointsConvertedPercentage", "BreakPointsOpportunities",
    ]

    def get_row(surf, yr):
        rows = player_df[
            (player_df["year"] == str(yr)) & (player_df["surface"] == surf)
        ]
        if rows.empty:
            return None
        row = rows.iloc[0]
        if row[stat_cols].isna().all():
            return None
        return row

    row = get_row(court_type, target_year)
    if row is None:
        print(f"Warning: no {court_type}/{target_year} data for {full_name}, trying {court_type}/all.")
        row = get_row(court_type, "all")
    if row is None:
        print(f"Warning: no {court_type}/all data for {full_name}, trying all/all.")
        row = get_row("all", "all")
    if row is None:
        raise ValueError(f"No stats found for {full_name} (tried {court_type}/{target_year}, {court_type}/all, all/all)")

    return {
        "first_in":        row["FirstServePercentage"] / 100,
        "win_first":       row["FirstServePointsWonPercentage"] / 100,
        "win_second":      row["SecondServePointsWonPercentage"] / 100,
        "return_first":    row["FirstServeReturnPointsWonPercentage"] / 100,
        "return_second":   row["SecondServeReturnPointsWonPercentage"] / 100,
        "bp_save_rate":    row["BreakPointsSavedPercentage"] / 100,
        "bp_save_faced":   int(row["BreakPointsFaced"]),
        "bp_convert_rate": row["BreakPointsConvertedPercentage"] / 100,
        "bp_convert_opps": int(row["BreakPointsOpportunities"]),
    }


CSV_DIR  = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else r"d:\CMU\Kalshi\Tennis Monte Carlo\atp\data"
p1_stats = load_player_stats(P1_NAME, COURT_TYPE, YEAR - 1, csv_dir=CSV_DIR)
p2_stats = load_player_stats(P2_NAME, COURT_TYPE, YEAR - 1, csv_dir=CSV_DIR)
print(f"Loaded stats for {P1_NAME}:", p1_stats)
print(f"Loaded stats for {P2_NAME}:", p2_stats)

In [27]:
p1 = Player(P1_NAME, p1_stats, prior_strength=50, lam=0.90)
p2 = Player(P2_NAME, p2_stats, prior_strength=50, lam=0.90)

N = 50_000
print(f"Running {N:,} simulations ({COURT_TYPE}, {YEAR-1} stats)...")
p1_win_prob, p1_se, p2_win_prob, p2_se = run_simulation(p1, p2, N=N)
print(f"{P1_NAME} win probability : {p1_win_prob:.4f} ± {p1_se:.4f}")
print(f"P1 95% CI : ({p1_win_prob - 1.96*p1_se:.4f}, {p1_win_prob + 1.96*p1_se:.4f})")
print("---------------------------------")
print(f"{P2_NAME} win probability : {p2_win_prob:.4f} ± {p2_se:.4f}")
print(f"P2 95% CI : ({p2_win_prob - 1.96*p2_se:.4f}, {p2_win_prob + 1.96*p2_se:.4f})")

Running 50,000 simulations (Clay, 2025 stats)...
Daniel Merida win probability : 0.8196 ± 0.0017
P1 95% CI : (0.8162, 0.8230)
---------------------------------
Tomas Barrios Vera win probability : 0.1804 ± 0.0017
P2 95% CI : (0.1770, 0.1838)


In [ ]:
# ── RANKING BLEND ──────────────────────────────────
BLEND_W = 0.5   # weight on Monte Carlo; (1-w) goes to ranking signal

ranking_files = glob.glob(os.path.join(CSV_DIR, "player_rankings_*.csv"))
if not ranking_files:
    raise FileNotFoundError(f"No player_rankings_*.csv found in {CSV_DIR}")
rankings_df = pd.read_csv(max(ranking_files, key=os.path.getmtime))

def get_ranking_points(name):
    row = rankings_df[rankings_df["full_name"].str.strip().str.lower() == name.strip().lower()]
    if row.empty:
        raise ValueError(f"Player '{name}' not found in rankings CSV")
    return float(row.iloc[0]["ranking_points"])

pts1 = get_ranking_points(P1_NAME)
pts2 = get_ranking_points(P2_NAME)

p_ranking_p1 = pts1 / (pts1 + pts2)
p_ranking_p2 = pts2 / (pts1 + pts2)

p_final_p1 = BLEND_W * p1_win_prob + (1 - BLEND_W) * p_ranking_p1
p_final_p2 = BLEND_W * p2_win_prob + (1 - BLEND_W) * p_ranking_p2

print(f"Ranking points — {P1_NAME}: {pts1:.0f}  |  {P2_NAME}:  {pts2:.0f}")
print(f"Ranking signal — {P1_NAME}: {p_ranking_p1:.4f}  |  {P2_NAME}: {p_ranking_p2:.4f}")
print()
print(f"Monte Carlo    — {P1_NAME}: {p1_win_prob:.4f}  |  {P2_NAME}: {p2_win_prob:.4f}")
print(f"Blended (w={BLEND_W}) — {P1_NAME}: {p_final_p1:.4f}  |  {P2_NAME}: {p_final_p2:.4f}")

Ranking points — Cristian Garin: 602  |  Jan Choinski: 526
Ranking signal — Cristian Garin: 0.5337  |  Jan Choinski: 0.4663

Monte Carlo    — Cristian Garin: 0.3823  |  Jan Choinski: 0.6177
Blended (w=0.5) — Cristian Garin: 0.4580  |  Jan Choinski: 0.5420
